# Human-in-the-loop

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = init_chat_model("openai:gpt-4.1")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal
from langgraph.types import interrupt, Command


class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]
    intent: str

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

intentClassifier_template = ChatPromptTemplate.from_messages([
    ("system",
     """You are an intent classifier for DummyCorp. Your job is to decide whether the user's query belongs to customer support or technical assistance.
     Make sure to :
     Return only one word: 'customerSupport' or 'technical'."""),
    ("human", "{input}")
])

technical_template = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a knowledgeable technical support assistant for DummyCorp. "
     "Help users with setup, errors, or technical troubleshooting in max 25 words."),
    ("human", "{input}")
])


customerSupport_chain = customerSupport_template | llm

intentClassifier_chain = intentClassifier_template | llm

technical_chain = technical_template | llm
    
# Agent - 1
def intentClassifierAgent(state: State):
    response = intentClassifier_chain.invoke(state["messages"])
    print("classified as :",response)
    return {"messages": [response], "intent":[response][-1].content}


# Agent - 2
def customerSupportAgent(state: State):
    print("Came inside customer support")
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}

# Agent - 3
def technicalAgent(state: State):
    print("Came inside technical")
    response = technical_chain.invoke(state["messages"])
    return {"messages": [response]}

def route_intent(state: State) -> Literal["customerSupport", "technical"]:
    """
    Routes the conversation to the correct agent based on detected intent.
    """
    intent = state.get("intent", "")
    print("intent is ", intent)
    if "technical" in intent:
        return "technical"
    return "customerSupport"


def human_review(state: State):
    print("---PAUSING FOR HUMAN REVIEW---")
    intent_to_review = state.get("intent", "")
    
    edited_intent = interrupt(
        {
            "message": "Please review and accept or reject.",
            "intent": intent_to_review
        }
    )
    if edited_intent=="accept":
        return {"intent": intent_to_review}
    else:
        return {"intent": "customerSupport"}
        
graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("human_review", human_review)
graph_builder.add_node("intentClassifier", intentClassifierAgent)
graph_builder.add_node("technical", technicalAgent)

graph_builder.add_edge(START, "intentClassifier")
graph_builder.add_edge("intentClassifier","human_review")

graph_builder.add_conditional_edges(
    "human_review",
    route_intent,
    ["customerSupport", "technical"]
)
graph_builder.add_edge("technical", END)
graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
def run_graph(topic: str):
    """Function to run the graph interactively."""
    
    # Start the graph
    initial_result = graph.invoke({"messages": topic}, config=config)
    
    if '__interrupt__' in initial_result:
        interrupt_payload = initial_result['__interrupt__'][0].value
        print(f"\nSystem Message: {interrupt_payload['message']}")
        print(f" Review: '{interrupt_payload['intent']}'")
        
        edited_intent = input("\nEnter your edited response: ")
        
        # Resume the graph with the human's input
        final_result = graph.invoke(Command(resume=edited_intent), config=config)

        print(final_result["messages"][-1].content)
    else:
        print("Graph finished without interruption.")

# Start the interactive part
user_topic = input("You :")
run_graph(user_topic)